In [51]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

pandas → Handle dataset
numpy → Numerical operations
TfidfVectorizer → Convert text to numbers
cosine_similarity → Content-based recommendations
KMeans → Clustering algorithm

In [52]:
#load dataset
df = pd.read_csv("../data/recommendation_dataset.csv")

df.head()

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre,Genre,Content
0,#Girlboss,Sophia Amoruso,4.5,2272.0,615.0,"Sorry, we just need to make sure you're not a ...",-1,-1,Unknown,"Sorry, we just need to make sure you're not a ..."
1,#Girlboss,Sophia Amoruso,4.5,2272.0,615.0,"Penguin presents the unabridged downloadable, ...",-1,-1,Unknown,"Penguin presents the unabridged downloadable, ..."
2,#TheRealCinderella: #BestFriendsForever Series...,Yesenia Vargas,4.3,179.0,586.0,\n\nOops!\nIt's rush hour and traffic is pilin...,-1,-1,Unknown,\n\nOops!\nIt's rush hour and traffic is pilin...
3,10 Bedtime Stories For Little Kids,div.,-1.0,263.0,376.0,\nBy completing your purchase you agree to Aud...,-1,-1,Unknown,\nBy completing your purchase you agree to Aud...
4,10 Essential Pieces of Literature,Khalil Gibran,-1.0,263.0,32.0,This Audiobook contains the following works:,87 hours and 44 minutes,",#759 in Audible Audiobooks & Originals (See T...",Classic Fiction,This Audiobook contains the following works: C...


In [53]:
#check dataset
print("Shape:", df.shape)

df.info()

Shape: (4288, 10)
<class 'pandas.DataFrame'>
RangeIndex: 4288 entries, 0 to 4287
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Book Name          4288 non-null   str    
 1   Author             4288 non-null   str    
 2   Rating             4288 non-null   float64
 3   Number of Reviews  4288 non-null   float64
 4   Price              4288 non-null   float64
 5   Description        4288 non-null   str    
 6   Listening Time     4288 non-null   str    
 7   Ranks and Genre    4288 non-null   str    
 8   Genre              4288 non-null   str    
 9   Content            4288 non-null   str    
dtypes: float64(3), str(7)
memory usage: 3.1 MB


In [54]:
#Check Missing Values
df.isnull().sum()

Book Name            0
Author               0
Rating               0
Number of Reviews    0
Price                0
Description          0
Listening Time       0
Ranks and Genre      0
Genre                0
Content              0
dtype: int64

In [55]:
df = df.drop_duplicates(
    subset=["Book Name"]
).reset_index(drop=True)

In [56]:
#TF-IDF Vectorization (NLP)
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(df["Content"])

In [57]:
#Check TF-IDF Shape
tfidf_matrix.shape

(3999, 14288)

In [58]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [59]:
#Check Shape
cosine_sim.shape

(3999, 3999)

In [60]:
#Create Book Indices
indices = pd.Series(
    df.index,
    index=df["Book Name"]
).drop_duplicates()

indices.head()

Book Name
#Girlboss                                                                 0
#TheRealCinderella: #BestFriendsForever Series, Book 1                    1
10 Bedtime Stories For Little Kids                                        2
10 Essential Pieces of Literature                                         3
10 Essential Success Mantras from the Bhagavad Gita (Rupa Quick Reads)    4
dtype: int64

In [61]:
print(df.shape)

(3999, 10)


In [62]:
#recommendation function
def recommend_books(title):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the same book
    sim_scores = sim_scores[1:6]

    book_indices = [i[0] for i in sim_scores]

    return df[
        ["Book Name", "Author", "Genre"]
    ].iloc[book_indices]

In [63]:
#est the Recommendation System
df[
    df["Book Name"].str.contains(
        "Atomic",
        case=False,
        na=False
    )
]["Book Name"]

309      Atomic Attraction: The Psychology of Attraction
310    Atomic Habits: An Easy and Proven Way to Build...
Name: Book Name, dtype: str

In [64]:
df.loc[310, "Book Name"]

'Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones'

In [65]:
recommend_books(
    "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"
)

,Book Name,Author,Genre
710,Decisive: How to Make Better Choices in Life a...,Chip Heath,Personal Success
896,Everything is Figureoutable: How One Simple Be...,Marie Forleo,Personal Success
1324,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Personal Success
3318,"The Power of Habit: Why We Do What We Do, and ...",Charles Duhigg,Personal Success
3679,"Thinking, Fast and Slow",Daniel Kahneman,Personal Success


In [66]:
#Train K-Means
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=50,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(tfidf_matrix)

In [67]:
#adds the cluster labels to your dataset.
df["Cluster"] = clusters

In [68]:
#hows how many books are present in each cluster.
df["Cluster"].value_counts()

Cluster
1     963
0     544
37    331
49     98
4      98
10     94
12     87
11     81
3      81
2      79
14     72
9      66
5      63
24     57
27     57
32     56
22     55
6      55
35     55
45     52
33     51
8      49
30     48
19     46
36     42
26     38
46     38
38     38
18     37
40     37
16     36
15     35
28     34
31     33
13     32
41     32
34     31
43     29
20     29
29     29
21     27
47     25
7      24
44     23
23     23
42     22
48     21
39     19
25     14
17     13
Name: count, dtype: int64

In [69]:
#Create Cluster Recommendation Function
def cluster_recommendations(title, n=5):
    # Find the cluster of the selected book
    cluster = df.loc[
        df["Book Name"] == title,
        "Cluster"
    ].values[0]

    # Get books from the same cluster
    recommendations = df[
        (df["Cluster"] == cluster) &
        (df["Book Name"] != title)
    ][["Book Name", "Author", "Genre"]]

    return recommendations.head(n)

In [70]:
#Test It
cluster_recommendations(
    "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"
)

,Book Name,Author,Genre
36,21 Lessons for the 21st Century,Yuval Noah Harari,Political Theory
73,A Brief History of Time: From Big Bang to Blac...,Stephen Hawking,Cosmology
90,A Delayed Life: The True Story of The Libraria...,Dita Kraus,Unknown
136,A Room of One's Own: Penguin Classics,Virginia Woolf,Society & Culture
269,Anxious People,Fredrik Backman,Humour


Content-Based Filtering was the best-performing approach for this Audible recommendation dataset.
"Although K-Means clustering was successfully implemented, the generated clusters did not produce highly relevant recommendations. Content-based filtering using TF-IDF and cosine similarity provided more meaningful results."

In [71]:
#Hybrid Function
def hybrid_recommendations(title, n=5):
    # Get index of selected book
    idx = indices[title]

    # Find cluster of selected book
    cluster = df.iloc[idx]["Cluster"]

    # Books in same cluster
    cluster_books = df[df["Cluster"] == cluster].index.tolist()

    # Similarity scores
    sim_scores = []

    for book_idx in cluster_books:
        if book_idx != idx:
            sim_scores.append((book_idx, cosine_sim[idx][book_idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Top N recommendations
    top_books = [i[0] for i in sim_scores[:n]]

    return df.loc[
        top_books,
        ["Book Name", "Author", "Genre"]
    ]

Hybrid
Atomic Habits
↓
Find books in same cluster
↓
Within that cluster,
use cosine similarity
↓
Top 5 recommendations

In [72]:
#Test It
hybrid_recommendations(
    "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"
)

,Book Name,Author,Genre
710,Decisive: How to Make Better Choices in Life a...,Chip Heath,Personal Success
896,Everything is Figureoutable: How One Simple Be...,Marie Forleo,Personal Success
1324,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Personal Success
3318,"The Power of Habit: Why We Do What We Do, and ...",Charles Duhigg,Personal Success
3679,"Thinking, Fast and Slow",Daniel Kahneman,Personal Success


Three recommendation approaches were implemented: Content-Based Filtering, K-Means Clustering, and a Hybrid Approach. The K-Means approach produced broader recommendations, whereas Content-Based Filtering generated more relevant suggestions. The Hybrid Approach, which combined cluster information with cosine similarity, produced the most meaningful recommendations for our dataset

In [73]:
#Save the Models
import joblib

In [74]:
#save TF-IDF Vectorizer
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [75]:
#Save Cosine Similarity Matrix
joblib.dump(
    cosine_sim,
    "../models/cosine_similarity.pkl"
)

['../models/cosine_similarity.pkl']

In [76]:
#Save K-Means Model
joblib.dump(
    kmeans,
    "../models/kmeans_model.pkl"
)

['../models/kmeans_model.pkl']

In [77]:
#Save Final Dataset
df.to_csv(
    "../models/final_books.csv",
    index=False
)

In [78]:
#evaluation
def evaluate_model(recommended, relevant):
    recommended = set(recommended)
    relevant = set(relevant)

    true_positive = len(recommended & relevant)

    precision = true_positive / len(recommended)
    recall = true_positive / len(relevant)

    return precision, recall

In [79]:
#Define the relevant books
relevant_books = [
    "Decisive: How to Make Better Choices in Life and Work",
    "Everything is Figureoutable: How One Simple Belief Can Help Us Overcome Any Obstacle and Create Unstoppable Success",
    "Ikigai: The Japanese Secret to a Long and Happy Life",
    "The Power of Habit: Why We Do What We Do, and How to Change",
    "Thinking, Fast and Slow"
]

In [80]:
#Define the Content-Based recommendations
content_books = [
    "Decisive: How to Make Better Choices in Life and Work",
    "Everything is Figureoutable: How One Simple Belief Can Help Us Overcome Any Obstacle and Create Unstoppable Success",
    "Ikigai: The Japanese Secret to a Long and Happy Life",
    "The Power of Habit: Why We Do What We Do, and How to Change",
    "Thinking, Fast and Slow"
]

In [81]:
#Calculate Precision and Recall
precision, recall = evaluate_model(content_books, relevant_books)

print("Precision@5:", precision)
print("Recall@5:", recall)

Precision@5: 1.0
Recall@5: 1.0


Since the dataset did not contain user interaction data, Precision@5 and Recall@5 were estimated using manually selected relevant recommendations for comparison purposes